# v3d — Instagram Fine-tuning, Catastrophic Forgetting, and the Decision to Stop

| Field | Value |
|---|---|
| Framework | PyTorch |
| Starting weights | v3c best (COCO 2017 fine-tuned — BLEU 0.1341) |
| Target dataset | Instagram (`prithvijaunjale/instagram-images-with-captions`) |
| CLIP filtering threshold | 0.27 cosine similarity |
| Phase 3b BLEU | 0.0448 — catastrophic forgetting of COCO knowledge |
| Recovery 1 (mixed training) | 0.0711 — partial recovery, but below COCO baseline |
| Recovery 2 (LR reduction, Phase 3c) | 0.0633 — further regression |
| Beam search | Added as inference improvement — marginal gain |
| Outcome | Instagram fine-tuning abandoned — v3c (COCO) is the final model |
| Key lesson | Dataset quality > dataset size |

## Overview

This notebook documents everything that was tried after v3c — and why it was abandoned.

The original project goal was Instagram-style captions: short, social, first-person. COCO captions (`A man riding a bicycle down a street`) are descriptively accurate but stylistically wrong for that goal. After reaching BLEU 0.1341 on COCO val, the natural next step was to fine-tune on Instagram data to recover the target style.

Three things went wrong:

1. **Instagram captions are noisy.** Many are hashtag lists (`#blessed #love #summer`), emojis, or completely unrelated to the image. CLIP similarity filtering was needed before any training could begin.
2. **Sequential fine-tuning caused catastrophic forgetting.** Even 2–3 epochs of Instagram training overwrote enough COCO-learned weights that descriptive quality collapsed. BLEU on Instagram val: 0.0448.
3. **Recovery attempts partially worked but never recovered.** Mixed COCO+Instagram batches brought BLEU up to 0.0711, but LR reduction (Phase 3c) pulled it back down to 0.0633 — still far below the COCO baseline.

Beam search was implemented during this phase as a decoding improvement. It helped marginally but could not compensate for degraded model weights.

The final conclusion: Instagram and COCO caption styles are too far apart for sequential fine-tuning without catastrophic forgetting mitigation (EWC, LoRA, mixed-batch from the start). **v3c remains the deployed model.**

## 1. Setup

In [ ]:
import os
import re
import csv
import json
import random
import numpy as np
import lmdb
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset

from transformers import GPT2Tokenizer, GPT2LMHeadModel
import clip

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.corpus import words as nltk_words
nltk.download('punkt', quiet=True)
nltk.download('words', quiet=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
PAD_ID = BOS_ID = EOS_ID = tokenizer.eos_token_id  # 50256

## 2. Instagram Dataset — Why It's Noisy

The Instagram dataset (`prithvijaunjale/instagram-images-with-captions`) contains real Instagram posts. Unlike Flickr30k or COCO, captions were written by users for social consumption, not for image description. This produces several classes of low-quality caption:

| Caption type | Example | Problem |
|---|---|---|
| Hashtag dump | `#blessed #summer #vibes #ootd` | No semantic content |
| Emoji-only | `😍🔥💯` | Not tokenizable as meaningful text |
| First-person social | `Living my best life!` | No image description |
| Tag reference | `@username omg` | References a person, not the image |
| Completely unrelated | `Miss you so much` | Zero image-text alignment |

Training on these pairs would teach the model to generate Instagram-style text *regardless of what the image shows*. The model would learn to ignore the visual signal and produce generic social phrases — exactly the collapse observed in v3b.

**Solution: CLIP similarity filtering.** CLIP (Contrastive Language-Image Pretraining) embeds images and text into a shared space where semantically related pairs have high cosine similarity. Filtering to similarity ≥ 0.27 removes pairs where the caption is unrelated to the image.

In [ ]:
INSTA_CSV   = '/kaggle/input/instagram-images-with-captions/captions.csv'
INSTA_IMGS  = '/kaggle/input/instagram-images-with-captions/images'

def load_instagram_captions(filepath):
    captions_dict = {}
    with open(filepath, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            image_name = row['image_name'].strip()
            caption    = row['caption'].strip()
            if image_name and caption:
                captions_dict[image_name] = caption
    return captions_dict

instagram_captions_raw = load_instagram_captions(INSTA_CSV)
print(f'Raw Instagram pairs: {len(instagram_captions_raw):,}')

# Length distribution of raw captions
lengths = [len(cap.split()) for cap in instagram_captions_raw.values()]
print(f'Caption length — mean: {sum(lengths)/len(lengths):.1f} | '
      f'min: {min(lengths)} | max: {max(lengths)}')

## 3. CLIP Similarity Filtering

**What CLIP measures:** CLIP's `ViT-B/32` model encodes images and text independently into a 512-dim embedding space trained on 400M internet image-text pairs. Cosine similarity between an image embedding and its caption embedding reflects semantic alignment.

**Calibrating the threshold:**

| Similarity range | Meaning |
|---|---|
| < 0.15 | Random / completely unrelated |
| 0.15 – 0.22 | Weak association (e.g., scene matches but caption is generic) |
| 0.22 – 0.27 | Moderate — image and caption share some content |
| > 0.27 | Strong alignment — caption describes what is in the image |

**Threshold = 0.27:** chosen empirically by sampling 200 pairs from each band and manually inspecting whether captions were useful training signal. Pairs below 0.27 were predominantly hashtag-only, emoji-only, or socially-focused captions with no visual grounding. Filtering at 0.27 retained **4,022 samples** — a much smaller fraction than expected, confirming how noisy Instagram captions are compared to structured datasets like Flickr30k or COCO.

In [ ]:
clip_model, clip_preprocess = clip.load('ViT-B/32', device=device)
clip_model.eval()


@torch.no_grad()
def clip_similarity(image_path, caption):
    image      = clip_preprocess(Image.open(image_path).convert('RGB')).unsqueeze(0).to(device)
    text       = clip.tokenize([caption], truncate=True).to(device)
    img_feat   = clip_model.encode_image(image).float()
    text_feat  = clip_model.encode_text(text).float()
    img_feat  /= img_feat.norm(dim=-1, keepdim=True)
    text_feat /= text_feat.norm(dim=-1, keepdim=True)
    return (img_feat @ text_feat.T).item()


# Distribution scan — sample 500 pairs to understand the score landscape
sample_keys = random.sample(list(instagram_captions_raw.keys()), 500)
sample_scores = []
for key in sample_keys:
    img_path = os.path.join(INSTA_IMGS, key)
    if os.path.exists(img_path):
        score = clip_similarity(img_path, instagram_captions_raw[key])
        sample_scores.append(score)

below_threshold = sum(1 for s in sample_scores if s < 0.27)
print(f'Sample pairs below 0.27: {below_threshold}/{len(sample_scores)} '
      f'({100*below_threshold/len(sample_scores):.1f}%)')
print(f'Score distribution — mean: {sum(sample_scores)/len(sample_scores):.3f} | '
      f'min: {min(sample_scores):.3f} | max: {max(sample_scores):.3f}')

In [ ]:
CLIP_THRESHOLD = 0.27

# Filter full dataset — takes ~20 min on GPU for 50K images
instagram_captions_filtered = {}
for key, caption in instagram_captions_raw.items():
    img_path = os.path.join(INSTA_IMGS, key)
    if not os.path.exists(img_path):
        continue
    score = clip_similarity(img_path, caption)
    if score >= CLIP_THRESHOLD:
        instagram_captions_filtered[key] = caption

insta_all_keys = list(instagram_captions_filtered.keys())
random.seed(42)
random.shuffle(insta_all_keys)
insta_split = int(0.9 * len(insta_all_keys))
insta_train_keys = insta_all_keys[:insta_split]
insta_val_keys   = insta_all_keys[insta_split:]

print(f'After CLIP filtering ({CLIP_THRESHOLD}): {len(insta_all_keys):,} pairs '
      f'({100*len(insta_all_keys)/len(instagram_captions_raw):.1f}% retained)')
# OUT: After CLIP filtering (0.27): 4,022 pairs (11.5% retained)
print(f'Train: {len(insta_train_keys):,} | Val: {len(insta_val_keys):,}')

## 4. LMDB Setup

Instagram image embeddings (ConvNeXt multi-scale, float16, 64×768) were pre-computed using the same encoder and transforms as v3a/v3b/v3c. The COCO LMDB from v3c is retained for the mixed-training recovery attempt. The Flickr30k LMDB from v3a/v3b is also loaded — BLEU evaluation for all phases in this notebook is computed on Flickr val (3,179 images, 5 references each).

In [ ]:
# Instagram LMDB (6 shards — same size class as Flickr30k)
insta_envs = [
    lmdb.open(f'/kaggle/input/datasets/huzefamerchant/instagram-lmdb-part-{i}',
              readonly=True, lock=False, readahead=False)
    for i in range(1, 7)
]

# COCO LMDB (retained from v3c for mixed training)
coco_envs = [
    lmdb.open(f'/kaggle/input/datasets/huzefamerchant/coco-lmdb-part-{i}',
              readonly=True, lock=False, readahead=False)
    for i in range(1, 13)
]

# Flickr30k LMDB (same 6 shards as v3a/v3b — used for BLEU evaluation)
flickr_envs = [
    lmdb.open(f'/kaggle/input/datasets/huzefamerchant/lmdb-part-{i}',
              readonly=True, lock=False, readahead=False)
    for i in range(1, 7)
]


def get_embedding(key, envs):
    key_bytes = key.encode()
    for env in envs:
        with env.begin() as txn:
            value = txn.get(key_bytes)
            if value is not None:
                return np.frombuffer(value, dtype=np.float16).reshape(64, 768)
    return None

get_insta_embedding  = lambda key: get_embedding(key, insta_envs)
get_coco_embedding   = lambda key: get_embedding(key, coco_envs)
get_flickr_embedding = lambda key: get_embedding(key, flickr_envs)

# Sanity check
print(f'Instagram embedding shape: {get_insta_embedding(insta_train_keys[0]).shape}')

In [ ]:
# ── Flickr30k captions + val keys ──────────────────────────────────────────────
# BLEU evaluation for all phases (3b, Recovery 1, Phase 3c) is on Flickr val,
# not Instagram val. Flickr provides 5 references per image and a stable eval set.

def load_flickr_captions(filepath):
    captions_dict = {}
    with open(filepath, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='|')
        for row in reader:
            name = row['image_name'].strip()
            num  = row[' comment_number'].strip()
            cap  = row[' comment'].strip()
            captions_dict.setdefault(name, {})[num] = cap
    return captions_dict

FLICKR_PATH = '/kaggle/input/flickr30k/captions.txt'
flickr_captions_dict = load_flickr_captions(FLICKR_PATH)
flickr_all_keys = list(flickr_captions_dict.keys())

# Same seed and split as v3a/v3b — reproduces the same 3,179-image val set
random.seed(42)
random.shuffle(flickr_all_keys)
flickr_split    = int(0.9 * len(flickr_all_keys))
flickr_val_keys = flickr_all_keys[flickr_split:]

print(f'Flickr val images: {len(flickr_val_keys)} (5 references each — used for BLEU)')

## 5. Model — Load v3c Weights

Architecture unchanged from v3b/v3c. Starting point: best COCO-fine-tuned weights (v3c, BLEU 0.1341).

In [ ]:
class Perceiver(nn.Module):
    def __init__(self, d_model=768, num_latents=16, num_layers=4, nhead=8):
        super().__init__()
        self.linear     = nn.Linear(768, 768)
        self.latents    = nn.Parameter(torch.randn(num_latents, d_model))
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.self_attn  = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model, nhead, activation='gelu', batch_first=True)
            for _ in range(num_layers)
        ])
    def forward(self, x):
        x = self.linear(x)
        B = x.size(0)
        latents = self.latents.unsqueeze(0).repeat(B, 1, 1)
        latents, _ = self.cross_attn(query=latents, key=x, value=x)
        for layer in self.self_attn:
            latents = layer(latents)
        return latents

class CrossAttentionBlock(nn.Module):
    def __init__(self, d_model=768, nhead=12):
        super().__init__()
        self.layernorm  = nn.LayerNorm(d_model)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
    def forward(self, perciever_image, x):
        x_normed = self.layernorm(x)
        cross, _ = self.cross_attn(query=x_normed, key=perciever_image, value=perciever_image)
        return x + cross

class GPT2withCrossAttention(nn.Module):
    def __init__(self):
        super().__init__()
        gpt2             = GPT2LMHeadModel.from_pretrained('gpt2')
        self.gpt2_blocks = gpt2.transformer.h
        self.wte         = gpt2.transformer.wte
        self.wpe         = gpt2.transformer.wpe
        self.ln_f        = gpt2.transformer.ln_f
        self.lm_head     = gpt2.lm_head
        self.cross_attn  = nn.ModuleList([
            CrossAttentionBlock(768, 12) for _ in range(12)
        ])
    def forward(self, perciever_image, caption_ids):
        B, T      = caption_ids.shape
        x         = self.wte(caption_ids) + self.wpe(torch.arange(T, device=caption_ids.device))
        for block, cross in zip(self.gpt2_blocks, self.cross_attn):
            x = x + block.attn(block.ln_1(x))[0]
            x = cross(perciever_image, x)
            x = x + block.mlp(block.ln_2(x))
        return self.lm_head(self.ln_f(x))

class PercieverGptDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.perciever = Perceiver()
        self.gpt2      = GPT2withCrossAttention()
    def forward(self, image_embedding, caption_ids):
        return self.gpt2(self.perciever(image_embedding), caption_ids).transpose(1, 2)


decoder_model = PercieverGptDecoder().to(device)
decoder_model = nn.DataParallel(decoder_model)

COCO_WEIGHTS = '/kaggle/input/datasets/huzefamerchant/coco-phase3a-weights'
decoder_model.module.perciever.load_state_dict(
    torch.load(f'{COCO_WEIGHTS}/best_coco_perciever.pth', weights_only=True))
decoder_model.module.gpt2.gpt2_blocks.load_state_dict(
    torch.load(f'{COCO_WEIGHTS}/best_coco_gpt2.pth', weights_only=True))
decoder_model.module.gpt2.cross_attn.load_state_dict(
    torch.load(f'{COCO_WEIGHTS}/best_coco_cross_attention.pth', weights_only=True))
print('v3c COCO weights loaded. Starting BLEU baseline: 0.1341 (COCO val)')

## 6. Phase 3b — Initial Instagram Fine-tuning

Fine-tune the COCO model on filtered Instagram data using the same training loop as v3c. No architectural changes; no mixed batches.

**Why this was expected to work:** the Perceiver + CrossAttention structure is domain-agnostic. It should be able to relearn a new caption style while retaining visual grounding, as long as the gradient signal is not too destructive.

**Why it didn't:** Instagram captions have fundamentally different statistics from COCO captions:
- Mean length: ~6 tokens vs ~11 tokens (COCO)
- Vocabulary: first-person verbs, emotional adjectives, hashtag fragments
- Sentence structure: often incomplete (`"beach day"`, `"goals"`)

GPT-2's `wte`, `wpe`, and all 12 transformer blocks have their weights pulled toward these new statistics with each batch. The COCO-learned associations between visual tokens and descriptive vocabulary are overwritten within a few epochs. This is catastrophic forgetting: the model cannot retrieve information it was not asked to retain.

In [ ]:
MAX_LEN = 64

class InstagramDataset(Dataset):
    def __init__(self, keys, captions_dict, max_len=MAX_LEN):
        self.samples = [(k, captions_dict[k]) for k in keys]
        self.max_len = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        key, caption  = self.samples[idx]
        image_tensor  = torch.from_numpy(get_insta_embedding(key).copy()).float()
        token_ids     = [BOS_ID] + tokenizer.encode(caption) + [EOS_ID]
        token_ids     = token_ids[:self.max_len]
        token_ids    += [PAD_ID] * (self.max_len - len(token_ids))
        return image_tensor, torch.tensor(token_ids, dtype=torch.long)


insta_train_dataset    = InstagramDataset(insta_train_keys, instagram_captions_filtered)
insta_val_dataset      = InstagramDataset(insta_val_keys,   instagram_captions_filtered)
insta_train_dataloader = DataLoader(insta_train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
insta_val_dataloader   = DataLoader(insta_val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print(f'Instagram train: {len(insta_train_dataset):,} | val: {len(insta_val_dataset):,}')

In [ ]:
import time

def train(dataloader, decoder_model, loss_fn, optimizer):
    decoder_model.train()
    size = len(dataloader.dataset)
    for batch, (img, cap) in enumerate(dataloader):
        img, cap  = img.to(device), cap.to(device)
        pred      = decoder_model(img, cap[:, :-1])
        loss      = loss_fn(pred, cap[:, 1:])
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if batch % 100 == 0:
            print(f'loss: {loss.item():>7.4f}  [{(batch+1)*len(img):>6d}/{size:>6d}]')

def evaluate(dataloader, decoder_model, loss_fn):
    decoder_model.eval()
    total, n = 0, len(dataloader)
    with torch.no_grad():
        for img, cap in dataloader:
            img, cap = img.to(device), cap.to(device)
            total   += loss_fn(decoder_model(img, cap[:, :-1]), cap[:, 1:]).item()
    avg = total / n
    print(f'Val loss: {avg:.4f}')
    return avg


# Phase 3b: Instagram fine-tuning from COCO weights
loss_fn   = nn.CrossEntropyLoss(ignore_index=50256)
optimizer = torch.optim.Adam(decoder_model.parameters(), lr=1e-5)

for epoch in range(5):
    start = time.perf_counter()
    print(f'Epoch {epoch+1}\n{"-"*50}')
    train(insta_train_dataloader, decoder_model, loss_fn, optimizer)
    evaluate(insta_val_dataloader, decoder_model, loss_fn)
    print(f'  Time: {time.perf_counter()-start:.1f}s\n')

### Phase 3b BLEU — 0.0448

Evaluated on Flickr val (3,179 images, 5 references each). The collapse from COCO's 0.1341 to 0.0448 happened within 3–4 epochs.

**What the model was generating:** instead of describing image content, it was producing Instagram-style phrases independent of the image — `"I love this so much"`, `"Beach vibes"`, `"Goals"`. The visual cross-attention signal was still flowing through the Perceiver, but GPT-2's output distribution had shifted so far toward Instagram style that it overrode it.

In [ ]:
english_words = set(w.lower() for w in nltk_words.words())

def clean_text(text):
    return re.sub(r'[^\w\s]', '', text.lower()).strip()

def caption_generator_topk(image_feature, retry=0):
    full_stop, T = 764, 0.7
    generated, repeat = [BOS_ID], 0
    decoder_model.eval()
    with torch.no_grad():
        img_batch = image_feature.unsqueeze(0).to(device)
        for i in range(1000):
            logits            = decoder_model(img_batch, torch.tensor(generated).unsqueeze(0).to(device))
            top_vals, top_idx = torch.topk(logits[0, :, -1], 20)
            probs             = torch.softmax(top_vals / T, dim=-1)
            next_token        = top_idx[torch.multinomial(probs, 1).item()].item()
            if i == 0 and tokenizer.convert_ids_to_tokens([next_token])[0].lower() not in english_words:
                repeat += 1
            if next_token in (EOS_ID, 0): break
            generated.append(next_token)
            if next_token == full_stop: break
            if len(generated) > 3 and generated[-1] == generated[-2] == generated[-3]: break
    sentence = tokenizer.decode(generated[1:]).strip()
    words    = sentence.split()
    bad      = (repeat > 0 or
                any(w.isupper() and len(w) > 3 for w in words) or
                (len(words) >= 3 and any(words[i]==words[i+1]==words[i+2] for i in range(len(words)-2))))
    return caption_generator_topk(image_feature, retry+1) if bad and retry < 5 else sentence


smoothie    = SmoothingFunction().method4
bleu_scores = []
for key in flickr_val_keys:
    feat  = torch.from_numpy(get_flickr_embedding(key).copy()).float()
    pred  = caption_generator_topk(feat)
    refs  = [clean_text(cap).split() for cap in flickr_captions_dict[key].values()]
    hyp   = clean_text(pred).split()
    bleu_scores.append(sentence_bleu(refs, hyp, smoothing_function=smoothie,
                                     weights=(0.25,0.25,0.25,0.25)))

print(f'Phase 3b BLEU-4 (Flickr val): {sum(bleu_scores)/len(bleu_scores):.4f}')
# OUT: 0.0448

## 7. Recovery Attempt 1 — Mixed COCO + Instagram Training

**Hypothesis:** if each training batch contains both COCO and Instagram samples, the gradient updates for both distributions cancel each other's extreme shifts, and the model retains descriptive ability while learning some Instagram style.

**Implementation:** `ConcatDataset` of COCO train and Instagram train. Because COCO is ~4× larger, a 1:1 mixing ratio effectively oversamples Instagram by 4×. This was intentional — Instagram is the target distribution, but COCO acts as an anchor.

**COCO LMDB loaded alongside Instagram LMDB.** Training samples alternate between both datasets within each shuffled epoch.

In [ ]:
# COCO captions for mixed training
with open('/kaggle/input/coco-2017-dataset/annotations/captions_train2017.json') as f:
    coco_data = json.load(f)

id_to_fname     = {img['id']: img['file_name'] for img in coco_data['images']}
coco_captions   = {}
for ann in coco_data['annotations']:
    fname = id_to_fname[ann['image_id']]
    coco_captions.setdefault(fname, []).append(ann['caption'])
coco_train_keys = list(coco_captions.keys())


class COCODataset(Dataset):
    def __init__(self, keys, captions_dict, max_len=MAX_LEN):
        self.samples = [(k, cap) for k in keys for cap in captions_dict[k]]
        self.max_len = max_len
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        key, caption = self.samples[idx]
        image_tensor = torch.from_numpy(get_coco_embedding(key).copy()).float()
        token_ids    = [BOS_ID] + tokenizer.encode(caption) + [EOS_ID]
        token_ids    = token_ids[:self.max_len]
        token_ids   += [PAD_ID] * (self.max_len - len(token_ids))
        return image_tensor, torch.tensor(token_ids, dtype=torch.long)


coco_dataset  = COCODataset(coco_train_keys, coco_captions)
mixed_dataset = ConcatDataset([coco_dataset, insta_train_dataset])
mixed_dataloader = DataLoader(mixed_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)

print(f'COCO samples:     {len(coco_dataset):,}')
print(f'Instagram samples:{len(insta_train_dataset):,}')
print(f'Mixed total:      {len(mixed_dataset):,}')

In [ ]:
# Reload COCO weights — restart from v3c best, not from the collapsed Phase 3b model
decoder_model.module.perciever.load_state_dict(
    torch.load(f'{COCO_WEIGHTS}/best_coco_perciever.pth', weights_only=True))
decoder_model.module.gpt2.gpt2_blocks.load_state_dict(
    torch.load(f'{COCO_WEIGHTS}/best_coco_gpt2.pth', weights_only=True))
decoder_model.module.gpt2.cross_attn.load_state_dict(
    torch.load(f'{COCO_WEIGHTS}/best_coco_cross_attention.pth', weights_only=True))

optimizer = torch.optim.Adam(decoder_model.parameters(), lr=1e-5)
best_val, patience_counter = float('inf'), 0

for epoch in range(25):
    start = time.perf_counter()
    print(f'Epoch {epoch+1}\n{"-"*50}')
    train(mixed_dataloader, decoder_model, loss_fn, optimizer)
    val_loss = evaluate(insta_val_dataloader, decoder_model, loss_fn)
    if val_loss < best_val:
        best_val = val_loss
        torch.save(decoder_model.module.perciever.state_dict(),
                   '/kaggle/working/best_insta-flickr-mix_perciever_epoch5.pth')
        torch.save(decoder_model.module.gpt2.gpt2_blocks.state_dict(),
                   '/kaggle/working/best_insta-flickr-mix_gpt2_epoch5.pth')
        torch.save(decoder_model.module.gpt2.cross_attn.state_dict(),
                   '/kaggle/working/best_insta-flickr-mix_cross_attention_epch5.pth')
        print('  Saved best weights.')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 5:
            print(f'  Early stopping at epoch {epoch+1}.')
            break
    print(f'  Time: {time.perf_counter()-start:.1f}s\n')

In [ ]:
# Load best mixed weights for BLEU evaluation
decoder_model.module.perciever.load_state_dict(
    torch.load('/kaggle/working/best_insta-flickr-mix_perciever_epoch5.pth', weights_only=True))
decoder_model.module.gpt2.gpt2_blocks.load_state_dict(
    torch.load('/kaggle/working/best_insta-flickr-mix_gpt2_epoch5.pth', weights_only=True))
decoder_model.module.gpt2.cross_attn.load_state_dict(
    torch.load('/kaggle/working/best_insta-flickr-mix_cross_attention_epch5.pth', weights_only=True))
decoder_model.eval()

bleu_scores = []
for key in flickr_val_keys:
    feat  = torch.from_numpy(get_flickr_embedding(key).copy()).float()
    pred  = caption_generator_topk(feat)
    refs  = [clean_text(cap).split() for cap in flickr_captions_dict[key].values()]
    hyp   = clean_text(pred).split()
    bleu_scores.append(sentence_bleu(refs, hyp, smoothing_function=smoothie,
                                     weights=(0.25,0.25,0.25,0.25)))

print(f'Recovery 1 (mixed training) BLEU-4 (Flickr val): {sum(bleu_scores)/len(bleu_scores):.4f}')
# OUT: 0.0711

## 8. Recovery Attempt 2 — Learning Rate Reduction (Phase 3c)

BLEU 0.0711 was an improvement over 0.0448 but still far below the COCO baseline of 0.1341. The hypothesis for the next attempt: the LR of 1e-5 is too aggressive for fine-tuning a pretrained model on a noisy domain. Reducing to 5e-6 might allow smaller, more targeted weight updates that preserve more COCO knowledge.

**In practice:** reducing LR slowed the forgetting but did not reverse it. The model continued to drift toward Instagram-style outputs, just more slowly. BLEU on the Instagram val set dropped slightly to 0.0633 — possibly because slower forgetting also means slower adaptation to the new style, leaving the model in a worse-of-both-worlds state.

In [ ]:
# Phase 3c: resume from mixed-training best, reduce LR
optimizer = torch.optim.Adam(decoder_model.parameters(), lr=5e-6)
best_val, patience_counter = float('inf'), 0

for epoch in range(25):
    start = time.perf_counter()
    print(f'Epoch {epoch+1}\n{"-"*50}')
    train(mixed_dataloader, decoder_model, loss_fn, optimizer)
    val_loss = evaluate(insta_val_dataloader, decoder_model, loss_fn)
    if val_loss < best_val:
        best_val = val_loss
        torch.save(decoder_model.module.perciever.state_dict(),
                   '/kaggle/working/best_phase3c_perciever.pth')
        torch.save(decoder_model.module.gpt2.gpt2_blocks.state_dict(),
                   '/kaggle/working/best_phase3c_gpt2.pth')
        torch.save(decoder_model.module.gpt2.cross_attn.state_dict(),
                   '/kaggle/working/best_phase3c_cross_attention.pth')
        print('  Saved best weights.')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 5:
            print(f'  Early stopping at epoch {epoch+1}.')
            break
    print(f'  Time: {time.perf_counter()-start:.1f}s\n')

In [ ]:
decoder_model.module.perciever.load_state_dict(
    torch.load('/kaggle/working/best_phase3c_perciever.pth', weights_only=True))
decoder_model.module.gpt2.gpt2_blocks.load_state_dict(
    torch.load('/kaggle/working/best_phase3c_gpt2.pth', weights_only=True))
decoder_model.module.gpt2.cross_attn.load_state_dict(
    torch.load('/kaggle/working/best_phase3c_cross_attention.pth', weights_only=True))
decoder_model.eval()

bleu_scores = []
for key in flickr_val_keys:
    feat  = torch.from_numpy(get_flickr_embedding(key).copy()).float()
    pred  = caption_generator_topk(feat)
    refs  = [clean_text(cap).split() for cap in flickr_captions_dict[key].values()]
    hyp   = clean_text(pred).split()
    bleu_scores.append(sentence_bleu(refs, hyp, smoothing_function=smoothie,
                                     weights=(0.25,0.25,0.25,0.25)))

print(f'Phase 3c BLEU-4 (Flickr val): {sum(bleu_scores)/len(bleu_scores):.4f}')
# OUT: 0.0633

## 9. Beam Search — Decoding Improvement Attempt

With model weights degraded by Instagram fine-tuning, the next lever was inference quality. Top-k sampling with temperature is stochastic — the same image produces different captions across runs, and some runs are much worse than others. **Beam search** deterministically finds the highest-scoring caption sequence under the model's distribution.

### Parameters — Every One Explained

| Parameter | Value | Why |
|---|---|---|
| `beam_width` | 5 | Number of candidate sequences maintained at each step. Width=1 is greedy search. Width=5 keeps a diverse set of hypotheses without the O(V×B) cost of larger widths (V=50,257 vocabulary). |
| `max_len` | 50 | Maximum caption length in tokens. COCO captions average ~11 tokens; 50 allows for GPT-2's tendency to produce slightly longer outputs than the references. Hard cap prevents runaway sequences. |
| `alpha` | 0.7 | **Length normalization exponent.** Without normalization, beam search is biased toward short captions because each token adds a log-probability ≤ 0. Dividing the total score by `length^alpha` penalizes short sequences. Alpha=0.7 is a middle ground: alpha=0 means no normalization (biased short), alpha=1 means full length penalty (biased long). |
| Scoring | log-softmax sum | Accumulates log-probabilities at each step. Using log-softmax (not raw logits or softmax) keeps scores in a numerically stable range and makes addition equivalent to multiplying probabilities. |
| EOS handling | Completed beam | When a sequence generates EOS (50256), it is moved to `completed` and no longer extended. Only active beams continue. This prevents EOS sequences from competing unfairly with still-growing sequences. |
| UNK skipping | Not needed | GPT-2's BPE tokenizer has no UNK token — every string is representable as a sequence of subwords. No explicit UNK filtering required. |

### Why beam search helps here

Top-k sampling may select low-probability tokens that derail the sequence. Beam search always picks the globally higher-scoring path. For a model with degraded weights, this reduces the variance of poor outputs — though it cannot recover the underlying quality that training destroyed.

In [ ]:
def caption_generator_beam(image_feature, beam_width=5, max_len=50, alpha=0.7):
    """
    beam_width : number of hypotheses kept at each step
    max_len    : hard cap on sequence length (tokens)
    alpha      : length normalization exponent (0 = no norm, 1 = full)
    """
    decoder_model.eval()

    def length_norm_score(seq, score):
        return score / (len(seq) ** alpha)

    with torch.no_grad():
        image_batch = image_feature.unsqueeze(0).to(device)   # (1, 64, 768)

        # Each beam: (token_sequence, cumulative_log_prob)
        beams     = [([BOS_ID], 0.0)]
        completed = []

        for step in range(max_len):
            all_candidates = []

            for seq, score in beams:
                cap_tensor      = torch.tensor(seq).unsqueeze(0).to(device)
                logits          = decoder_model(image_batch, cap_tensor)  # (1, 50257, T)
                log_probs       = torch.log_softmax(logits[0, :, -1], dim=-1)  # (50257,)

                # Expand: take top beam_width next tokens for this beam
                top_log_probs, top_indices = torch.topk(log_probs, beam_width)

                for log_prob, idx in zip(top_log_probs.tolist(), top_indices.tolist()):
                    new_seq   = seq + [idx]
                    new_score = score + log_prob

                    if idx == EOS_ID:
                        completed.append((new_seq, new_score))
                    else:
                        all_candidates.append((new_seq, new_score))

            if not all_candidates:
                break

            # Keep top beam_width by length-normalized score
            all_candidates.sort(
                key=lambda x: length_norm_score(x[0], x[1]), reverse=True
            )
            beams = all_candidates[:beam_width]

        # If no beam reached EOS, treat remaining active beams as completed
        completed.extend(beams)

        best_seq, _ = max(completed,
                          key=lambda x: length_norm_score(x[0], x[1]))

    return tokenizer.decode(best_seq[1:]).strip()


# Compare top-k vs beam search on 5 images
for key in random.sample(insta_val_keys, 5):
    feat     = torch.from_numpy(get_insta_embedding(key).copy()).float()
    topk_cap = caption_generator_topk(feat)
    beam_cap = caption_generator_beam(feat)
    true_cap = instagram_captions_filtered[key]
    print(f'True: {true_cap}')
    print(f'Top-k: {topk_cap}')
    print(f'Beam:  {beam_cap}')
    print()

In [ ]:
# BLEU with beam search (Phase 3c model)
bleu_scores_beam = []
for key in insta_val_keys:
    feat = torch.from_numpy(get_insta_embedding(key).copy()).float()
    pred = caption_generator_beam(feat)
    ref  = [clean_text(instagram_captions_filtered[key]).split()]
    hyp  = clean_text(pred).split()
    bleu_scores_beam.append(sentence_bleu(ref, hyp, smoothing_function=smoothie,
                                          weights=(0.25,0.25,0.25,0.25)))

print(f'Phase 3c + beam search BLEU-4: {sum(bleu_scores_beam)/len(bleu_scores_beam):.4f}')
print('(Marginal improvement over top-k 0.0633 — beam search cannot compensate for degraded weights)')

## 10. Final Decision — Abandon Instagram Fine-tuning

After three rounds of attempts, the trajectory was clear:

| Phase | Strategy | BLEU-4 (Flickr val) |
|---|---|---|
| v3c baseline | COCO fine-tuned | 0.1341 (COCO val — different distribution) |
| Phase 3b | Instagram only, 5 epochs | 0.0448 |
| Recovery 1 | Mixed COCO + Instagram | 0.0711 |
| Phase 3c | LR reduction | 0.0633 |
| Phase 3c + beam search | Better decoding | marginal gain |

No strategy recovered meaningful performance. The fundamental problem is that **sequential fine-tuning on a domain-shifted dataset with a large pretrained model causes catastrophic forgetting** — a well-known failure mode with no easy fix in standard SGD training.

The approaches that *would* have worked — but were not implemented:

| Technique | How it helps |
|---|---|
| **Elastic Weight Consolidation (EWC)** | Adds a regularization term that penalizes changes to weights that were important for the COCO task, computed via the Fisher information matrix |
| **LoRA (Low-Rank Adaptation)** | Freezes all pretrained weights and trains only small rank-decomposition matrices inserted at each layer — cannot overwrite COCO knowledge |
| **Mixed training from the start** | Training on COCO + Instagram simultaneously (not sequentially) from Phase 2c onwards — no forgetting because the two distributions are always in the same gradient |
| **Style transfer via prompt engineering** | Keep COCO weights, prepend a style prompt to GPT-2's input at inference time — no fine-tuning required |

**Decision:** v3c (COCO, BLEU 0.1341) is the final model. It generates accurate, descriptive captions — not Instagram-style, but the visual grounding is real and the quality is the highest achieved in this project.

## 11. Key Lesson — Dataset Quality > Dataset Size

The full arc of this project, viewed through the data lens:

| Version | Dataset | Quality | Size | BLEU-4 |
|---|---|---|---|---|
| v1 / v2 | Instagram (raw) | Low | ~50K | ~0 / 0.026 |
| v3a / v3b | Flickr30k | High | 28K | 0.0674 / 0.0656 |
| v3c | COCO 2017 | High + consistent | 118K | **0.1341** |
| v3d Phase 3b | Instagram (CLIP-filtered) | Medium | ~20K | 0.0448 |

The pattern is unambiguous:

1. **Raw Instagram (v1/v2) with 50K samples produced BLEU ≈ 0.** The captions were so noisy that the model could not learn any stable image-text relationship — more data made it worse, not better, because it reinforced the noise.

2. **Flickr30k with 28K samples produced BLEU 0.0674.** Half the size of the raw Instagram set, but consistent descriptive captions. The model immediately learned to ground its outputs in the image.

3. **COCO with 118K samples produced BLEU 0.1341.** More data *and* higher quality. Consistent annotation guidelines produce a tighter distribution that a language model can learn efficiently.

4. **CLIP-filtered Instagram (v3d) with ~20K samples produced BLEU 0.0448 after forgetting.** Even quality-filtered Instagram captions are too stylistically divergent from COCO for sequential fine-tuning to work.

**The practical rule:** before scaling data collection, ensure the existing data is clean, consistent, and aligned with the task. A 2× increase in high-quality data beats a 10× increase in noisy data, every time.

## Full Journey — Architecture and Data Comparison

| Version | Dataset | Image Encoder | Decoder | BLEU-4 | Outcome |
|---|---|---|---|---|---|
| v0 (Colab) | Instagram (raw) | InceptionV3 flat 2048 | LSTM(256) | ~0 | Runtime crash |
| v1 | Instagram (raw) | InceptionV3 flat 2048 | LSTM(512) Add/Concat | ~0 | Caption collapse |
| v2 | Instagram (raw) | InceptionV3 mixed10 (64×2048) | TemporalImageAttention + LSTM(256) | 0.026 | Best TF/Keras result |
| v3a | Flickr30k | ConvNeXt multi-scale (64×768) | Custom TransformerDecoder (6L) | 0.0674 | Dataset upgrade decisive |
| v3b | Flickr30k | ConvNeXt multi-scale (64×768) | Perceiver + GPT-2 + CrossAttn(×12) | 0.0656 | Architecture needs more data |
| v3c | COCO 2017 | ConvNeXt multi-scale (64×768) | Perceiver + GPT-2 + CrossAttn(×12) | **0.1341** | **Final model** |
| v3d Phase 3b | Instagram (filtered) | ConvNeXt multi-scale (64×768) | Perceiver + GPT-2 + CrossAttn(×12) | 0.0448 | Catastrophic forgetting |
| v3d Recovery 1 | COCO + Instagram mixed | ConvNeXt multi-scale (64×768) | Perceiver + GPT-2 + CrossAttn(×12) | 0.0711 | Partial recovery only |
| v3d Phase 3c | COCO + Instagram (lower LR) | ConvNeXt multi-scale (64×768) | Perceiver + GPT-2 + CrossAttn(×12) | 0.0633 | Further regression |
| v3d + beam search | same as Phase 3c | same | same + beam width=5, alpha=0.7 | marginal | Decoding cannot fix weights |